In [2]:
from pathlib import Path
import pandas as pd

rna_dir = Path("../data/raw/rna")

rna_files = list(
    rna_dir.rglob("*.rna_seq.augmented_star_gene_counts.tsv")
)

print("RNA files:", len(rna_files))
rna_files[:3]

RNA files: 541


[WindowsPath('../data/raw/rna/000d81dd-9ba4-4852-9090-2bf22f6483f0/09755ce8-ed89-411a-a42f-b3edc4e41eeb.rna_seq.augmented_star_gene_counts.tsv'),
 WindowsPath('../data/raw/rna/00174492-1adc-4310-8147-a622befbb466/84e40c7e-2f60-4461-b14c-0859697295a4.rna_seq.augmented_star_gene_counts.tsv'),
 WindowsPath('../data/raw/rna/00ddbdd1-f29d-438b-820b-aacb111a3ec4/e635dbc7-ef8d-45e2-acd2-b961f0c8b600.rna_seq.augmented_star_gene_counts.tsv')]

In [3]:
sample_file = rna_files[0]

sample_rna = pd.read_csv(
    sample_file,
    sep="\t",
    comment="#"
)

print(sample_file)
print(sample_rna.shape)
print(sample_rna.columns.tolist())

sample_rna.head()

..\data\raw\rna\000d81dd-9ba4-4852-9090-2bf22f6483f0\09755ce8-ed89-411a-a42f-b3edc4e41eeb.rna_seq.augmented_star_gene_counts.tsv
(60664, 9)
['gene_id', 'gene_name', 'gene_type', 'unstranded', 'stranded_first', 'stranded_second', 'tpm_unstranded', 'fpkm_unstranded', 'fpkm_uq_unstranded']


,gene_id,gene_name,gene_type,unstranded,stranded_first,stranded_second,tpm_unstranded,fpkm_unstranded,fpkm_uq_unstranded
0,N_unmapped,NaN,NaN,1814468,1814468,1814468,NaN,NaN,NaN
1,N_multimapping,NaN,NaN,3294784,3294784,3294784,NaN,NaN,NaN
2,N_noFeature,NaN,NaN,4053713,22898782,22748008,NaN,NaN,NaN
3,N_ambiguous,NaN,NaN,3953132,915908,921288,NaN,NaN,NaN
4,ENSG00000000003.15,TSPAN6,protein_coding,2157,1097,1060,50.3093,13.6366,13.0426


In [4]:
for col in sample_rna.columns:
    print(col)

gene_id
gene_name
gene_type
unstranded
stranded_first
stranded_second
tpm_unstranded
fpkm_unstranded
fpkm_uq_unstranded


In [5]:
gene_rows = sample_rna[
    sample_rna["gene_id"].astype(str).str.startswith("ENSG")
].copy()

print("전체 행:", len(sample_rna))
print("실제 gene 행:", len(gene_rows))

gene_rows.head()

전체 행: 60664
실제 gene 행: 60660


,gene_id,gene_name,gene_type,unstranded,stranded_first,stranded_second,tpm_unstranded,fpkm_unstranded,fpkm_uq_unstranded
4,ENSG00000000003.15,TSPAN6,protein_coding,2157,1097,1060,50.3093,13.6366,13.0426
5,ENSG00000000005.6,TNMD,protein_coding,26,14,12,1.8636,0.5051,0.4831
6,ENSG00000000419.13,DPM1,protein_coding,978,499,480,85.7241,23.2360,22.2239
7,ENSG00000000457.14,SCYL3,protein_coding,604,452,465,9.2839,2.5165,2.4068
8,ENSG00000000460.17,C1orf112,protein_coding,182,268,266,3.2253,0.8742,0.8362


In [6]:
gene_rows["gene_type"].value_counts().head(20)

gene_type
protein_coding                        19962
lncRNA                                16901
processed_pseudogene                  10167
unprocessed_pseudogene                 2614
misc_RNA                               2212
snRNA                                  1901
miRNA                                  1881
TEC                                    1057
snoRNA                                  943
transcribed_unprocessed_pseudogene      939
transcribed_processed_pseudogene        500
rRNA_pseudogene                         497
IG_V_pseudogene                         187
IG_V_gene                               145
transcribed_unitary_pseudogene          138
TR_V_gene                               106
unitary_pseudogene                       98
TR_J_gene                                79
scaRNA                                   49
polymorphic_pseudogene                   48
Name: count, dtype: int64

In [7]:
protein_coding = gene_rows[
    gene_rows["gene_type"] == "protein_coding"
].copy()

print("protein-coding genes:", len(protein_coding))

protein_coding[
    ["gene_id", "gene_name", "tpm_unstranded"]
].head()

protein-coding genes: 19962


,gene_id,gene_name,tpm_unstranded
4,ENSG00000000003.15,TSPAN6,50.3093
5,ENSG00000000005.6,TNMD,1.8636
6,ENSG00000000419.13,DPM1,85.7241
7,ENSG00000000457.14,SCYL3,9.2839
8,ENSG00000000460.17,C1orf112,3.2253


In [8]:
print("TPM 결측치:", protein_coding["tpm_unstranded"].isna().sum())
print("TPM 최소값:", protein_coding["tpm_unstranded"].min())
print("TPM 최대값:", protein_coding["tpm_unstranded"].max())

protein_coding["tpm_unstranded"].describe()

TPM 결측치: 0
TPM 최소값: 0.0
TPM 최대값: 38151.6508


count    19962.000000
mean        46.511838
std        465.693330
min          0.000000
25%          0.345325
50%          8.684450
75%         31.129075
max      38151.650800
Name: tpm_unstranded, dtype: float64

In [9]:
print("gene_name 결측:", protein_coding["gene_name"].isna().sum())
print("고유 gene_name:", protein_coding["gene_name"].nunique())
print("중복 gene_name:", protein_coding["gene_name"].duplicated().sum())

gene_name 결측: 0
고유 gene_name: 19938
중복 gene_name: 24


In [10]:
duplicated_genes = protein_coding[
    protein_coding["gene_name"].duplicated(keep=False)
].sort_values("gene_name")

duplicated_genes[
    ["gene_id", "gene_name", "tpm_unstranded"]
]

,gene_id,gene_name,tpm_unstranded
15283,ENSG00000182584.6,ACTL10,0.0522
60647,ENSG00000288649.1,ACTL10,0.0000
17595,ENSG00000197976.12,AKAP17A,42.6825
17596,ENSG00000197976.12_PAR_Y,AKAP17A,0.0000
17108,ENSG00000196433.13,ASMT,0.2959
17109,ENSG00000196433.13_PAR_Y,ASMT,0.0000
12516,ENSG00000169093.16,ASMTL,37.0559
12517,ENSG00000169093.16_PAR_Y,ASMTL,0.0000
27,ENSG00000002586.20,CD99,94.0583
28,ENSG00000002586.20_PAR_Y,CD99,0.0000


In [11]:
print("gene_id 고유 개수:", protein_coding["gene_id"].nunique())
print("gene_id 중복:", protein_coding["gene_id"].duplicated().sum())

gene_id 고유 개수: 19962
gene_id 중복: 0


In [12]:
n_pc = len(protein_coding)
n_unique_symbol = protein_coding["gene_name"].nunique()
n_dup_rows = protein_coding["gene_name"].duplicated().sum()

print("Protein-coding rows:", n_pc)
print("Unique gene symbols:", n_unique_symbol)
print("Duplicated symbol rows:", n_dup_rows)

Protein-coding rows: 19962
Unique gene symbols: 19938
Duplicated symbol rows: 24


### Gene annotation audit

- Protein-coding gene rows: 19,962
- Unique gene symbols: 19,938
- Missing gene symbols: 0
- Duplicated gene-symbol rows: 24
- Duplicates include PAR_Y annotations and multiple Ensembl IDs mapped to the same symbol.
- Gene-level modeling will retain Ensembl gene IDs.
- Pathway-level analysis will collapse duplicated gene symbols using a predefined aggregation rule.

In [13]:
manifest_path = Path("../data/manifests/gdc_manifest_kirc_rna.txt")

manifest = pd.read_csv(
    manifest_path,
    sep="\t"
)

print(manifest.shape)
print(manifest.columns.tolist())
manifest.head()

(541, 5)
['id', 'filename', 'md5', 'size', 'state']


,id,filename,md5,size,state
0,fc63bae4-fdfb-4990-9902-3e0fb51ec092,e95ef864-eeee-4883-943c-e2e1eecd259c.rna_seq.a...,86038fbe25da92aadb11c62bd2ae30fb,4224305,released
1,4a29721a-1de3-4e14-b905-f39aacb1bd53,1fced4a1-6992-4189-84d9-fddce3e3b937.rna_seq.a...,9eaa62903907b3001326a9237901176b,4260833,released
2,862e5c87-bed8-4310-b9ed-1018568bdc78,7d160562-b94b-4cd9-9905-d15dadbb394a.rna_seq.a...,ee37e0c5d82097175f355b8cdd2aaeff,4259740,released
3,813cb150-7ef6-4fa7-9a64-c70f9401513c,6834bc84-328c-49be-b78e-61ffec46ef97.rna_seq.a...,308a4ca0dec476c0e45aabc4f9871b9f,4247961,released
4,4ea0bf83-1d7e-41da-a34d-221add8155b2,7afb81f7-f9b0-4d0b-a4bd-18c15043c73f.rna_seq.a...,a1a95aa249c908c63610172e15757344,4253724,released


In [14]:
import requests
from io import StringIO

files_endpt = "https://api.gdc.cancer.gov/files"

payload = {
    "filters": {
        "op": "in",
        "content": {
            "field": "files.file_id",
            "value": manifest["id"].tolist()
        }
    },
    "format": "tsv",
    "fields": ",".join([
        "file_id",
        "file_name",
        "cases.submitter_id",
        "cases.samples.submitter_id",
        "cases.samples.sample_type",
        "cases.samples.tissue_type",
        "cases.samples.tumor_descriptor",
        "cases.project.project_id"
    ]),
    "size": "1000"
}

response = requests.post(files_endpt, json=payload)
response.raise_for_status()

rna_metadata = pd.read_csv(
    StringIO(response.text),
    sep="\t"
)

print(rna_metadata.shape)
print(rna_metadata.columns.tolist())

rna_metadata.head()

(541, 9)
['cases.0.project.project_id', 'cases.0.samples.0.sample_type', 'cases.0.samples.0.submitter_id', 'cases.0.samples.0.tissue_type', 'cases.0.samples.0.tumor_descriptor', 'cases.0.submitter_id', 'file_id', 'file_name', 'id']


,cases.0.project.project_id,cases.0.samples.0.sample_type,cases.0.samples.0.submitter_id,cases.0.samples.0.tissue_type,cases.0.samples.0.tumor_descriptor,cases.0.submitter_id,file_id,file_name,id
0,TCGA-KIRC,Primary Tumor,TCGA-BP-4160-01A,Tumor,Primary,TCGA-BP-4160,fc63bae4-fdfb-4990-9902-3e0fb51ec092,e95ef864-eeee-4883-943c-e2e1eecd259c.rna_seq.a...,fc63bae4-fdfb-4990-9902-3e0fb51ec092
1,TCGA-KIRC,Primary Tumor,TCGA-BP-4965-01A,Tumor,Primary,TCGA-BP-4965,4a29721a-1de3-4e14-b905-f39aacb1bd53,1fced4a1-6992-4189-84d9-fddce3e3b937.rna_seq.a...,4a29721a-1de3-4e14-b905-f39aacb1bd53
2,TCGA-KIRC,Primary Tumor,TCGA-BP-4999-01A,Tumor,Primary,TCGA-BP-4999,862e5c87-bed8-4310-b9ed-1018568bdc78,7d160562-b94b-4cd9-9905-d15dadbb394a.rna_seq.a...,862e5c87-bed8-4310-b9ed-1018568bdc78
3,TCGA-KIRC,Primary Tumor,TCGA-BP-5009-01A,Tumor,Primary,TCGA-BP-5009,813cb150-7ef6-4fa7-9a64-c70f9401513c,6834bc84-328c-49be-b78e-61ffec46ef97.rna_seq.a...,813cb150-7ef6-4fa7-9a64-c70f9401513c
4,TCGA-KIRC,Primary Tumor,TCGA-CJ-4894-01A,Tumor,Primary,TCGA-CJ-4894,4ea0bf83-1d7e-41da-a34d-221add8155b2,7afb81f7-f9b0-4d0b-a4bd-18c15043c73f.rna_seq.a...,4ea0bf83-1d7e-41da-a34d-221add8155b2


In [15]:
for col in rna_metadata.columns:
    print(col)

cases.0.project.project_id
cases.0.samples.0.sample_type
cases.0.samples.0.submitter_id
cases.0.samples.0.tissue_type
cases.0.samples.0.tumor_descriptor
cases.0.submitter_id
file_id
file_name
id


In [16]:
rna_meta = rna_metadata.rename(columns={
    "cases.0.project.project_id": "project_id",
    "cases.0.samples.0.sample_type": "sample_type",
    "cases.0.samples.0.submitter_id": "sample_id",
    "cases.0.samples.0.tissue_type": "tissue_type",
    "cases.0.samples.0.tumor_descriptor": "tumor_descriptor",
    "cases.0.submitter_id": "patient_id",
    "file_id": "file_id",
    "file_name": "file_name"
})[
    [
        "file_id",
        "file_name",
        "patient_id",
        "sample_id",
        "sample_type",
        "tissue_type",
        "tumor_descriptor",
        "project_id"
    ]
].copy()

rna_meta.head()

,file_id,file_name,patient_id,sample_id,sample_type,tissue_type,tumor_descriptor,project_id
0,fc63bae4-fdfb-4990-9902-3e0fb51ec092,e95ef864-eeee-4883-943c-e2e1eecd259c.rna_seq.a...,TCGA-BP-4160,TCGA-BP-4160-01A,Primary Tumor,Tumor,Primary,TCGA-KIRC
1,4a29721a-1de3-4e14-b905-f39aacb1bd53,1fced4a1-6992-4189-84d9-fddce3e3b937.rna_seq.a...,TCGA-BP-4965,TCGA-BP-4965-01A,Primary Tumor,Tumor,Primary,TCGA-KIRC
2,862e5c87-bed8-4310-b9ed-1018568bdc78,7d160562-b94b-4cd9-9905-d15dadbb394a.rna_seq.a...,TCGA-BP-4999,TCGA-BP-4999-01A,Primary Tumor,Tumor,Primary,TCGA-KIRC
3,813cb150-7ef6-4fa7-9a64-c70f9401513c,6834bc84-328c-49be-b78e-61ffec46ef97.rna_seq.a...,TCGA-BP-5009,TCGA-BP-5009-01A,Primary Tumor,Tumor,Primary,TCGA-KIRC
4,4ea0bf83-1d7e-41da-a34d-221add8155b2,7afb81f7-f9b0-4d0b-a4bd-18c15043c73f.rna_seq.a...,TCGA-CJ-4894,TCGA-CJ-4894-01A,Primary Tumor,Tumor,Primary,TCGA-KIRC


In [17]:
patient_col = "cases.0.submitter_id"
sample_col = "cases.0.samples.0.submitter_id"

print("RNA 파일 수:", len(rna_metadata))
print("고유 환자 수:", rna_metadata[patient_col].nunique())
print("고유 sample 수:", rna_metadata[sample_col].nunique())

print("\n환자별 RNA 파일 개수")
print(
    rna_metadata[patient_col]
    .value_counts()
    .value_counts()
    .sort_index()
)

RNA 파일 수: 541
고유 환자 수: 533
고유 sample 수: 537

환자별 RNA 파일 개수
count
1    529
3      4
Name: count, dtype: int64


In [18]:
patient_counts = rna_metadata[patient_col].value_counts()

duplicate_patients = patient_counts[patient_counts > 1]

print("RNA 파일이 여러 개인 환자 수:", len(duplicate_patients))
duplicate_patients

RNA 파일이 여러 개인 환자 수: 4


cases.0.submitter_id
TCGA-B2-3923    3
TCGA-B2-5633    3
TCGA-B2-3924    3
TCGA-B2-5635    3
Name: count, dtype: int64

In [19]:
print(rna_meta["sample_type"].value_counts(dropna=False))
print()
print(rna_meta["tissue_type"].value_counts(dropna=False))
print()
print(rna_meta["tumor_descriptor"].value_counts(dropna=False))

sample_type
Primary Tumor    541
Name: count, dtype: int64

tissue_type
Tumor    541
Name: count, dtype: int64

tumor_descriptor
Primary    541
Name: count, dtype: int64


In [20]:
patient_counts = rna_meta["patient_id"].value_counts()

duplicate_patient_ids = patient_counts[
    patient_counts > 1
].index

duplicate_rna = (
    rna_meta[
        rna_meta["patient_id"].isin(duplicate_patient_ids)
    ]
    .sort_values(["patient_id", "sample_id", "file_id"])
)

duplicate_rna[
    [
        "patient_id",
        "sample_id",
        "sample_type",
        "tissue_type",
        "tumor_descriptor",
        "file_id",
        "file_name"
    ]
]

,patient_id,sample_id,sample_type,tissue_type,tumor_descriptor,file_id,file_name
234,TCGA-B2-3923,TCGA-B2-3923-01A,Primary Tumor,Tumor,Primary,2b393323-81e3-49e2-9acd-18eb8ab0c563,d96fc441-a213-4f24-b2b1-a23d63883532.rna_seq.a...
41,TCGA-B2-3923,TCGA-B2-3923-01A,Primary Tumor,Tumor,Primary,4134fa64-6365-4a43-b990-48342be44405,2a0d7ffd-21a2-4c9f-b420-9f78e9b8689b.rna_seq.a...
114,TCGA-B2-3923,TCGA-B2-3923-01B,Primary Tumor,Tumor,Primary,d514c3bc-c1e1-4028-9822-90ad703a3a8d,d5340889-fb1c-411d-922c-a9f69e1ac405.rna_seq.a...
197,TCGA-B2-3924,TCGA-B2-3924-01A,Primary Tumor,Tumor,Primary,056c67e7-03b3-40cc-8c19-d6938ad634aa,343ac8e1-8afa-4e36-a373-ca28c3e7a713.rna_seq.a...
132,TCGA-B2-3924,TCGA-B2-3924-01A,Primary Tumor,Tumor,Primary,df630d23-f007-4cc4-9910-edb44046d556,a40c1d70-1750-45d0-a85b-688331b029ea.rna_seq.a...
522,TCGA-B2-3924,TCGA-B2-3924-01B,Primary Tumor,Tumor,Primary,3324e6df-28f9-42d8-8465-0fcc2d618004,f08c47f7-7b61-46a2-a961-d5fced8ebde5.rna_seq.a...
227,TCGA-B2-5633,TCGA-B2-5633-01A,Primary Tumor,Tumor,Primary,61894bd0-9fdb-4bb1-b32c-f52dbc10c850,1e28caa3-197f-4deb-95c1-f6d8a41756ca.rna_seq.a...
99,TCGA-B2-5633,TCGA-B2-5633-01A,Primary Tumor,Tumor,Primary,bdb74506-8766-42e1-922e-e03da4637a70,dc6421de-439b-4764-87bf-c1922cfeba3c.rna_seq.a...
100,TCGA-B2-5633,TCGA-B2-5633-01B,Primary Tumor,Tumor,Primary,307fc027-0530-452b-8fa9-d46f7fc73d0e,9cc9f5b7-bbbb-4d3b-8d98-9bf4425a214b.rna_seq.a...
218,TCGA-B2-5635,TCGA-B2-5635-01A,Primary Tumor,Tumor,Primary,b98e59fc-120a-45c4-aade-3b59ca4a1a86,c276c214-acd9-44b3-b932-971c77c4fa91.rna_seq.a...


In [21]:
duplicate_rna.groupby("patient_id").agg(
    n_files=("file_id", "nunique"),
    n_samples=("sample_id", "nunique")
)

,n_files,n_samples
patient_id,,
TCGA-B2-3923,3,2
TCGA-B2-3924,3,2
TCGA-B2-5633,3,2
TCGA-B2-5635,3,2


In [22]:
def get_library_size(file_id):
    file_path = next(
        (rna_dir / file_id).glob("*.rna_seq.augmented_star_gene_counts.tsv")
    )

    df = pd.read_csv(
        file_path,
        sep="\t",
        comment="#",
        usecols=["gene_id", "unstranded"]
    )

    # STAR summary 행 제거하고 실제 gene만 사용
    df = df[df["gene_id"].str.startswith("ENSG", na=False)]

    return df["unstranded"].sum()


duplicate_rna = duplicate_rna.copy()

duplicate_rna["library_size"] = duplicate_rna["file_id"].apply(
    get_library_size
)

duplicate_rna[
    [
        "patient_id",
        "sample_id",
        "file_id",
        "library_size"
    ]
].sort_values(
    ["patient_id", "library_size"],
    ascending=[True, False]
)

,patient_id,sample_id,file_id,library_size
234,TCGA-B2-3923,TCGA-B2-3923-01A,2b393323-81e3-49e2-9acd-18eb8ab0c563,46492604
41,TCGA-B2-3923,TCGA-B2-3923-01A,4134fa64-6365-4a43-b990-48342be44405,44200794
114,TCGA-B2-3923,TCGA-B2-3923-01B,d514c3bc-c1e1-4028-9822-90ad703a3a8d,21912888
197,TCGA-B2-3924,TCGA-B2-3924-01A,056c67e7-03b3-40cc-8c19-d6938ad634aa,54948226
522,TCGA-B2-3924,TCGA-B2-3924-01B,3324e6df-28f9-42d8-8465-0fcc2d618004,26730126
132,TCGA-B2-3924,TCGA-B2-3924-01A,df630d23-f007-4cc4-9910-edb44046d556,23442239
99,TCGA-B2-5633,TCGA-B2-5633-01A,bdb74506-8766-42e1-922e-e03da4637a70,50013779
227,TCGA-B2-5633,TCGA-B2-5633-01A,61894bd0-9fdb-4bb1-b32c-f52dbc10c850,43887947
100,TCGA-B2-5633,TCGA-B2-5633-01B,307fc027-0530-452b-8fa9-d46f7fc73d0e,26648529
218,TCGA-B2-5635,TCGA-B2-5635-01A,b98e59fc-120a-45c4-aade-3b59ca4a1a86,55661891


In [23]:
# 환자당 RNA 파일이 1개뿐인 환자
single_patient_ids = patient_counts[
    patient_counts == 1
].index

single_rna = rna_meta[
    rna_meta["patient_id"].isin(single_patient_ids)
].copy()


# RNA 파일이 여러 개인 4명:
# library size가 가장 큰 파일 1개 선택
selected_duplicates = (
    duplicate_rna
    .sort_values(
        ["patient_id", "library_size", "file_id"],
        ascending=[True, False, True]
    )
    .drop_duplicates(
        subset="patient_id",
        keep="first"
    )
)

# 최종 환자당 RNA 파일 1개
rna_selected = pd.concat(
    [
        single_rna,
        selected_duplicates[rna_meta.columns]
    ],
    ignore_index=True
)

print("선택된 RNA 파일 수:", len(rna_selected))
print("고유 환자 수:", rna_selected["patient_id"].nunique())
print("환자 ID 중복:", rna_selected["patient_id"].duplicated().sum())

선택된 RNA 파일 수: 533
고유 환자 수: 533
환자 ID 중복: 0


In [24]:
selected_duplicates[
    [
        "patient_id",
        "sample_id",
        "file_id",
        "library_size"
    ]
]

,patient_id,sample_id,file_id,library_size
234,TCGA-B2-3923,TCGA-B2-3923-01A,2b393323-81e3-49e2-9acd-18eb8ab0c563,46492604
197,TCGA-B2-3924,TCGA-B2-3924-01A,056c67e7-03b3-40cc-8c19-d6938ad634aa,54948226
99,TCGA-B2-5633,TCGA-B2-5633-01A,bdb74506-8766-42e1-922e-e03da4637a70,50013779
218,TCGA-B2-5635,TCGA-B2-5635-01A,b98e59fc-120a-45c4-aade-3b59ca4a1a86,55661891


In [25]:
clinical = pd.read_parquet(
    "../data/interim/kirc_clinical_pfi.parquet"
)

print("Clinical 환자 수:", len(clinical))
print("Clinical 고유 환자 수:", clinical["patient_id"].nunique())

print("RNA 환자 수:", len(rna_selected))
print("RNA 고유 환자 수:", rna_selected["patient_id"].nunique())

Clinical 환자 수: 535
Clinical 고유 환자 수: 535
RNA 환자 수: 533
RNA 고유 환자 수: 533


In [26]:
clinical_ids = set(clinical["patient_id"])
rna_ids = set(rna_selected["patient_id"])

common_ids = clinical_ids & rna_ids

print("Clinical ∩ RNA 공통 환자 수:", len(common_ids))
print("Clinical에만 있는 환자:", len(clinical_ids - rna_ids))
print("RNA에만 있는 환자:", len(rna_ids - clinical_ids))

Clinical ∩ RNA 공통 환자 수: 531
Clinical에만 있는 환자: 4
RNA에만 있는 환자: 2


In [27]:
print("Clinical에만 있는 환자")
print(sorted(clinical_ids - rna_ids))

print("\nRNA에만 있는 환자")
print(sorted(rna_ids - clinical_ids))

Clinical에만 있는 환자
['TCGA-A3-3336', 'TCGA-AK-3430', 'TCGA-CJ-4913', 'TCGA-CW-6096']

RNA에만 있는 환자
['TCGA-BP-5201', 'TCGA-CJ-4641']


In [28]:
cdr = pd.read_excel(
    "../data/raw/clinical/TCGA-CDR-SupplementalTableS1.xlsx"
)

excluded_ids = sorted(
    (clinical_ids - rna_ids) |
    (rna_ids - clinical_ids)
)

exclusion_audit = cdr[
    (cdr["type"] == "KIRC") &
    (cdr["bcr_patient_barcode"].isin(excluded_ids))
][
    [
        "bcr_patient_barcode",
        "PFI",
        "PFI.time"
    ]
]

exclusion_audit

,bcr_patient_barcode,PFI,PFI.time
3902,TCGA-A3-3336,0.0,1043.0
3941,TCGA-AK-3430,0.0,480.0
4264,TCGA-BP-5201,1.0,NaN
4273,TCGA-CJ-4641,1.0,NaN
4312,TCGA-CJ-4913,1.0,538.0
4351,TCGA-CW-6096,0.0,2701.0


In [29]:
final_cohort = clinical.merge(
    rna_selected[
        [
            "patient_id",
            "sample_id",
            "file_id",
            "file_name"
        ]
    ],
    on="patient_id",
    how="inner"
)

print("최종 분석 환자 수:", len(final_cohort))
print("고유 환자 수:", final_cohort["patient_id"].nunique())
print("환자 ID 중복:", final_cohort["patient_id"].duplicated().sum())

print("\nPFI event 분포")
print(final_cohort["pfi_event"].value_counts())

final_cohort.head()

최종 분석 환자 수: 531
고유 환자 수: 531
환자 ID 중복: 0

PFI event 분포
pfi_event
0.0    372
1.0    159
Name: count, dtype: int64


,patient_id,age,gender,stage,grade,pfi_event,pfi_time,sample_id,file_id,file_name
0,TCGA-3Z-A93Z,69.0,MALE,Stage I,G2,0.0,385.0,TCGA-3Z-A93Z-01A,21702617-18af-41b6-98fa-54395bfd0d9a,c3b9c2b0-bd3d-43ff-a17f-797122d5ac3c.rna_seq.a...
1,TCGA-6D-AA2E,68.0,FEMALE,Stage I,G2,0.0,362.0,TCGA-6D-AA2E-01A,44dade70-3ecc-4be0-88a8-772cb763d5bb,e4497835-a45f-488e-905c-b06d578efef8.rna_seq.a...
2,TCGA-A3-3306,67.0,MALE,Stage I,G3,0.0,1120.0,TCGA-A3-3306-01A,c1215d68-5b15-44c3-9d25-73a77b0b5f31,55599384-574d-44e9-aa61-96d27b54aff1.rna_seq.a...
3,TCGA-A3-3307,66.0,MALE,Stage III,G3,0.0,1436.0,TCGA-A3-3307-01A,b605224a-cf9c-4092-beb4-5ed43c1179de,78855102-69a0-454f-b67a-c0919dc154fb.rna_seq.a...
4,TCGA-A3-3308,77.0,FEMALE,Stage III,G2,0.0,16.0,TCGA-A3-3308-01A,7b34b9d4-8526-4738-87b4-f9435a1c7ae1,598575d0-6dc4-4d59-be17-5dff01e6ca52.rna_seq.a...


In [31]:
# ==========================================
# Final cohort / RNA selection QC
# ==========================================

import numpy as np

# ------------------------------------------
# 1. GDC metadata QC
# ------------------------------------------

assert len(rna_meta) == 541, \
    f"RNA metadata 파일 수 이상: {len(rna_meta)}"

assert rna_meta["file_id"].nunique() == 541, \
    "중복 file_id가 존재합니다."

assert (rna_meta["project_id"] == "TCGA-KIRC").all(), \
    "TCGA-KIRC가 아닌 파일이 포함되어 있습니다."

assert (rna_meta["sample_type"] == "Primary Tumor").all(), \
    "Primary Tumor가 아닌 sample이 포함되어 있습니다."

assert (rna_meta["tissue_type"] == "Tumor").all(), \
    "Tumor가 아닌 tissue가 포함되어 있습니다."

assert (rna_meta["tumor_descriptor"] == "Primary").all(), \
    "Primary가 아닌 tumor가 포함되어 있습니다."


# ------------------------------------------
# 2. 환자당 대표 RNA 선택 QC
# ------------------------------------------

assert len(rna_selected) == 533, \
    f"선택된 RNA 파일 수 이상: {len(rna_selected)}"

assert rna_selected["patient_id"].nunique() == 533, \
    "환자 수와 RNA 선택 수가 일치하지 않습니다."

assert rna_selected["patient_id"].duplicated().sum() == 0, \
    "환자당 RNA 파일이 2개 이상 남아 있습니다."

assert rna_selected["file_id"].duplicated().sum() == 0, \
    "동일 RNA file_id가 두 번 이상 선택되었습니다."


# ------------------------------------------
# 3. 중복 환자 대표 sample 선택 QC
# 최대 raw library size가 선택되었는지 확인
# ------------------------------------------

max_library_size = (
    duplicate_rna
    .groupby("patient_id")["library_size"]
    .max()
    .sort_index()
)

selected_library_size = (
    selected_duplicates
    .set_index("patient_id")["library_size"]
    .sort_index()
)

assert max_library_size.index.equals(
    selected_library_size.index
), "중복 환자 목록이 서로 일치하지 않습니다."

assert np.allclose(
    max_library_size.to_numpy(),
    selected_library_size.to_numpy()
), "중복 환자에서 최대 library size 파일이 선택되지 않았습니다."


# ------------------------------------------
# 4. Final cohort QC
# ------------------------------------------

assert len(final_cohort) == 531, \
    f"최종 cohort 환자 수 이상: {len(final_cohort)}"

assert final_cohort["patient_id"].nunique() == 531, \
    "최종 cohort의 고유 환자 수가 531명이 아닙니다."

assert final_cohort["patient_id"].duplicated().sum() == 0, \
    "최종 cohort에 중복 환자가 있습니다."

assert final_cohort["file_id"].duplicated().sum() == 0, \
    "최종 cohort에서 동일 RNA file이 중복 사용되고 있습니다."


# final_cohort의 RNA가 실제 rna_selected에서 선택된 파일인지 확인
assert set(final_cohort["file_id"]).issubset(
    set(rna_selected["file_id"])
), "final_cohort에 rna_selected에 없는 RNA 파일이 존재합니다."


# ------------------------------------------
# 5. Survival outcome QC
# ------------------------------------------

assert final_cohort["pfi_event"].notna().all(), \
    "pfi_event 결측치가 있습니다."

assert final_cohort["pfi_time"].notna().all(), \
    "pfi_time 결측치가 있습니다."

assert set(
    final_cohort["pfi_event"].unique()
).issubset({0, 1, 0.0, 1.0}), \
    "pfi_event에 0/1 이외의 값이 있습니다."

assert np.isfinite(
    final_cohort["pfi_time"]
).all(), \
    "pfi_time에 inf 또는 비정상 숫자가 있습니다."

assert (
    final_cohort["pfi_time"] >= 0
).all(), \
    "음수 pfi_time이 존재합니다."


# ------------------------------------------
# QC 결과 요약
# ------------------------------------------

print("===== FINAL QC PASSED =====")
print("GDC RNA files :", len(rna_meta))
print("RNA patients  :", len(rna_selected))
print("Final cohort  :", len(final_cohort))

print("\nPFI event distribution:")
print(
    final_cohort["pfi_event"]
    .value_counts()
    .sort_index()
)

print("\nPFI time summary:")
print(
    final_cohort["pfi_time"]
    .describe()
)

===== FINAL QC PASSED =====
GDC RNA files : 541
RNA patients  : 533
Final cohort  : 531

PFI event distribution:
pfi_event
0.0    372
1.0    159
Name: count, dtype: int64

PFI time summary:
count     531.000000
mean     1160.487759
std       946.407941
min         0.000000
25%       358.000000
50%      1011.000000
75%      1756.000000
max      4074.000000
Name: pfi_time, dtype: float64


In [32]:
zero_pfi = final_cohort[
    final_cohort["pfi_time"] == 0
].copy()

print("PFI time = 0 환자 수:", len(zero_pfi))

print(
    zero_pfi[
        [
            "patient_id",
            "pfi_event",
            "pfi_time",
            "stage",
            "grade"
        ]
    ]
)

PFI time = 0 환자 수: 2
      patient_id  pfi_event  pfi_time    stage grade
49  TCGA-A3-A8OU        0.0       0.0  Stage I    G1
52  TCGA-A3-A8OX        0.0       0.0  Stage I    G3


In [30]:
# from pathlib import Path

# output_path = Path(
#     "../data/interim/kirc_final_cohort.parquet"
# )

# final_cohort.to_parquet(
#     output_path,
#     index=False
# )

# print("저장 완료:", output_path)